# CS 3110/3990: Data Privacy
## In-Class Exercise #4, week of 9/28/2026

In [2]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

For various values of $b$, write code to print out the percent error of summing the ages in the `adult` dataset, 
clipped to each value of $b$.

In [3]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    print(f"% error for {b} is {pct_error(real_sum, clipped_sum)}")

% error for 1 is 97.4080940444511
% error for 11 is 71.48903448896206
% error for 21 is 46.000380495392264
% error for 31 is 25.509031989473492
% error for 41 is 11.7959939725709
% error for 51 is 4.329687317165198
% error for 61 is 1.1587597123836921
% error for 71 is 0.2341877497996031
% error for 81 is 0.03860675005194001
% error for 91 is 0.0


What value of $b$ is the best?

Any value of b > than 81 is fine, at b = 81 the relative error due to bias is small. Values much larger than this risk adding attitional error when we add noise to achieve DP. 

## Question 2

For various values of $b$, print the result of a *differentially private* sum of ages, clipped to each value of $b$. Use $\epsilon = 0.1$.

In [4]:
bs = range(1, 100, 10)
real_sum = adult['Age'].sum()

# If pct_error is removed, DP is satisfied
# For a total privacy cost of ep = 10 * 0.1 by sequential compostion
for b in bs:
    clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
    dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
    print(f"fDP result for {b} is {dp_sum}, error is {pct_error(real_sum, dp_sum)}")

fDP result for 1 is 32549.501412011035, error is 97.40900934983758
fDP result for 11 is 358036.4412996891, error is 71.49974556960167
fDP result for 21 is 678263.9057527558, error is 46.00914416773353
fDP result for 31 is 936358.5840968965, error is 25.46440862841787
fDP result for 41 is 1107439.4608672096, error is 11.846106261122557
fDP result for 51 is 1201920.326734716, error is 4.325283223519067
fDP result for 61 is 1241888.6226827095, error is 1.1437450551352566
fDP result for 71 is 1253231.1059960446, error is 0.24086584225643434
fDP result for 81 is 1255115.5090778195, error is 0.09086444271995858
fDP result for 91 is 1255344.1287587814, error is 0.07266596255532454


Which value of $b$ is the best now? Does it differ?

It does not really differ, 81 or 91 is still good, the error due to bias is much greater than the error due to variance, until the clipping param is really big

## Question 3

Write an algorithm to *automatically pick the clipping parameter* for the age column. Your algorithm should satisfy differential privacy.


Look for a bunch of canidates for b, like Q2
Find thte first pair(b, b') st noisy val for b > noisy val for b'

In [5]:
def pick_b(epsilon):
    bs = range(1, 200, 20)
    real_sum = adult['Age'].sum()
    epsilon_i = epsilon / len(bs)
    dp_sums = []
    last_sum = 0
    for b in bs: #Satisfies len(bs) * epsilon_i-DP by sequential comp
        clipped_sum = adult['Age'].clip(lower=0, upper = b).sum()
        dp_sum = laplace_mech(clipped_sum, sensitivity=b, epsilon=epsilon_i)
        if dp_sum < last_sum:
            return b
        else:
            last_sum = dp_sum
        # dp_sum=laplace_mech(clipped_sum, sensitivity=b, epsilon=0.1)
        # print(f"fDP result for {b} is {dp_sum}")
    
    # raise Exception("No good val!")
    return bs[-1]

pick_b(1.0)
result = laplace_mech(adult['Age'].clip(lower=0, upper= b), sensitivity=b, epsilon=2.0)

In [6]:
# TEST CASE for question 3

many_trials = np.mean([pick_b(1.0) for _ in range(100)])
assert many_trials > 70
assert many_trials < 500

## Question 4

What is the privacy cost of your algorithm, and why?

The privacy cost of pick_b is 'len(bs) * epsilon_i = epsilon'

Each iter fof the loop uses ep i privacy cost and there are len(bs) iters

## Question 5

Write code to generate a *histogram* of education numbers in the `adult` dataset.

In [8]:
def education_hist():
    hist = adult['Education'].value_counts()
    return hist


education_hist()

Education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

In [9]:
# TEST CASE for question 5
h = education_hist()
assert h['HS-grad'] == 10501
assert h['12th'] == 433
assert h['Doctorate'] == 413

## Question 6

Write code to release a *differentially private histogram* of education numbers.

In [12]:
def dp_education_hist(epsilon):
    hist = adult['Education'].value_counts()
    noisy_hist = hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    return noisy_hist
    # total cost is epsilon by parallel comp
    # The elements of the histrogram come from disjoint subsets of the data

dp_education_hist(1.0)

Education
HS-grad         10501.510495
Some-college     7290.729668
Bachelors        5352.249148
Masters          1721.981339
Assoc-voc        1384.224028
11th             1176.630334
Assoc-acdm       1068.234614
10th              932.212444
7th-8th           647.452092
Prof-school       578.778729
9th               511.849698
12th              433.805309
Doctorate         412.202503
5th-6th           332.887020
1st-4th           165.990991
Preschool          51.099540
Name: count, dtype: float64

In [13]:
# TEST CASE for question 6
h = dp_education_hist(1.0)
assert abs(h['HS-grad'] - 10501) < 100
assert abs(h['Doctorate'] - 413) < 100

## Question 7

What is the total privacy cost of `dp_education_hist`, and why?

Sensitivity: 1 countng query
Composition: epsilon by parallel composition
post proc: all ements elements satisfiy DP

## Question 8

Write code to generate a differentially private *contingency table* for the Education and Sex columns of the `adult` dataset.

In [ ]:
def dp_crosstab_education_sex(epsilon):
    hist = adult.groupby(['Education', 'Sex']).size()
    # This histogram groups by a combinations of values
    # Each row has exactly one combination of age and sex values
    # (Parallel composition there fore applies)
    return hist.apply(lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon))
    # If using df, same noise is used for every noise, hist does not have this issue
    # Total privacy cost is epsilon, by parallel compositon

dp_crosstab_education_sex(1.0)

Education     Sex   
10th          Female     294.389218
              Male       638.721550
11th          Female     431.470664
              Male       742.493380
12th          Female     145.199441
              Male       288.970703
1st-4th       Female      46.299355
              Male       121.539481
5th-6th       Female      83.994626
              Male       249.114694
7th-8th       Female     159.367002
              Male       487.766895
9th           Female     143.714944
              Male       370.316126
Assoc-acdm    Female     421.538806
              Male       642.100481
Assoc-voc     Female     500.872830
              Male       881.180968
Bachelors     Female    1617.490743
              Male      3737.668544
Doctorate     Female      84.358641
              Male       327.072133
HS-grad       Female    3387.982508
              Male      7110.934422
Masters       Female     535.486021
              Male      1188.103281
Preschool     Female      15.827198
       

In [ ]:
# TEST CASE for question 8
ct = dp_crosstab_education_sex(1.0)
assert abs(ct['Female']['10th'] - 295) < 100
assert abs(ct['Male']['10th'] - 638) < 100
assert abs(ct['Female']['Bachelors'] - 1619) < 100

assert abs(ct['Female']['10th'] - 295) > 0
assert abs(ct['Male']['10th'] - 638) > 0
assert abs(ct['Female']['Bachelors'] - 1619) > 0

## Question 9

Does parallel composition apply for the contingency table in question 1? Why or why not?

Yes we can! The chunks returned by the groupby to create the hist are disjoint, groupby puts rows into groups based on combinations of attribute values, each row has exactly one combination of attribute values.

## Question 10

Does the number of columns used in constructing the contingency table matter for privacy cost? Does it matter for accuracy?

For privacy cost is does not matter. You can use parallel comp for an n-dimensional histogram, for any value of n.